In [1]:
import sys
sys.path.append('..')
from osp import *

In [3]:
def format_url(url):
    x = url.split(".")
    return ".".join(x[1:])


def jstor2row(row, prefix='other', discipline='Other'):
    uuid = row['item_id']
    id = prefix + "/" + row['url'].split('/stable/')[-1]
    return {
        'id': id,
        "uuid": uuid,
        'title': row['title'],
        'author': row['creators_string'],
        'year': int(row['published_date'].split('-')[0]),
        'journal': row['is_part_of'],
        'volume': row['issue_volume'],
        'issue': row['issue_number'],
        'url': format_url(row['url']),
        'publisher': '; '.join(row['publishers']) if row['publishers'] else None,
        'discipline': discipline,
        'discipline_names': ' / '.join(row['discipline_names']) if row['discipline_names'] else None,
    }


def iter_jstor_other():
    item_ids = {d['iid'] for d in iter_jsonl(FN_JSTOR_DATA_OTHER)}
    for d in iter_jstor():
        if d['item_id'] in item_ids:
            yield jstor2row(d, prefix='other')


In [4]:
next(iter_jstor_other())

32277it [00:01, 18063.11it/s]
  0%|          | 374/12412004 [00:00<08:53, 23259.88it/s]


{'id': 'other/10.2307/25512149',
 'uuid': '850d2e3e-0e81-3542-9fa3-90d55da6c122',
 'title': 'Martha Randall and Acton National School',
 'author': None,
 'year': 2003,
 'journal': '"Before I Forget...": Journal of the Poyntzpass and District Local History Society',
 'volume': None,
 'issue': '9',
 'url': 'jstor.org/stable/10.2307/25512149',
 'publisher': 'Poyntzpass and District Local History Society',
 'discipline': 'Other',
 'discipline_names': 'History / Irish Studies'}

In [6]:
ld = list(iter_jstor_other())
df = pd.DataFrame(ld).fillna('')

32277it [00:01, 18005.68it/s]
100%|██████████| 12412004/12412004 [00:32<00:00, 384958.98it/s]


In [7]:
uuid2id = dict(zip(df['uuid'], df['id']))

In [8]:
len(uuid2id)

32277

In [ ]:
def save_jstor_other_txt():
    for d in iter_jsonl(FN_JSTOR_DATA_OTHER):
        id = uuid2id.get(d['iid'])
        if id:
            fn = '../data/raw/txt/' + id + '.txt'
            fldr = os.path.dirname(fn)
            os.makedirs(fldr, exist_ok=True)
            
            txt = '\n\n\n'.join(d['full_text'])
            with open(fn, 'w') as f:
                f.write(txt)

In [10]:
save_jstor_other_txt()

32277it [00:05, 6423.69it/s]


In [11]:
df1 = pd.read_csv('../data/metadata1.csv').fillna('')
df1

# df.set_index('id').to_csv('../data/metadata.csv')

,id,uuid,title,author,year,journal,volume,issue,url,publisher,discipline
0,phil/10.2307/40231690,f6eecd30-8c4a-3c3d-a9da-4e777d091b2c,"""Aristotlés"" Horror Vacui",John Thorp,1990,Canadian Journal of Philosophy,20,2,jstor.org/stable/10.2307/40231690,Cambridge University Press,Philosophy
1,phil/10.2307/40230399,aaff574b-da8a-389b-b82c-8bcfbf981a8e,"""None in Particular""",John Woods,1973,Canadian Journal of Philosophy,2,3,jstor.org/stable/10.2307/40230399,Cambridge University Press,Philosophy
2,phil/10.2307/40231533,9a99d15f-bbe2-349e-b5af-cf96fd35c6eb,"""Tractatus"" 2.022-2.023",Raymond D. Bradley,1987,Canadian Journal of Philosophy,17,2,jstor.org/stable/10.2307/40231533,Cambridge University Press,Philosophy
3,phil/10.2307/40230622,9af0da08-8130-3178-86cb-5df65d6fb0cd,"""Tractatus"" 5.54-5.5422",Eric B. Dayton,1976,Canadian Journal of Philosophy,6,2,jstor.org/stable/10.2307/40230622,Cambridge University Press,Philosophy
4,phil/10.2307/40231225,041ecb2f-f8ab-398e-a1df-8813eaffa841,"'Can,' Compatibilism, and Possible Worlds",Michael J. Zimmerman,1981,Canadian Journal of Philosophy,11,4,jstor.org/stable/10.2307/40231225,Cambridge University Press,Philosophy
...,...,...,...,...,...,...,...,...,...,...,...
58121,lit/2873182,,The Subversive Discourse of the Wife of Bath: Phallocentric Discourse and the Imprisonment of Criticism,Barrie Ruth Straus,1988,ELH,55,3,jstor.org/stable/2873182,Johns Hopkins University Press,Literature
58122,lit/459445,9a9690d3-bcdb-3274-8ec1-8e50da89d1bf,"Shelley's ""Hymn to Intellectual Beauty""",Elizabeth Nitchie,1948,PMLA,63,2,jstor.org/stable/459445,Modern Language Association,Literature
58123,lit/3715352,,"The Water-Bridge in Chrétien's ""Charrette""",K. G. T. Webster,1931,The Modern Language Review,26,1,jstor.org/stable/3715352,Modern Humanities Research Association,Literature
58124,lit/2871890,,The Creation of the Self in Gerard Manley Hopkins,J. Hillis Miller,1955,ELH,22,4,jstor.org/stable/2871890,Johns Hopkins University Press,Literature


In [12]:
df2 = pd.concat([df1, df]).fillna('')
df2

,id,uuid,title,author,year,journal,volume,issue,url,publisher,discipline,discipline_names
0,phil/10.2307/40231690,f6eecd30-8c4a-3c3d-a9da-4e777d091b2c,"""Aristotlés"" Horror Vacui",John Thorp,1990,Canadian Journal of Philosophy,20,2,jstor.org/stable/10.2307/40231690,Cambridge University Press,Philosophy,
1,phil/10.2307/40230399,aaff574b-da8a-389b-b82c-8bcfbf981a8e,"""None in Particular""",John Woods,1973,Canadian Journal of Philosophy,2,3,jstor.org/stable/10.2307/40230399,Cambridge University Press,Philosophy,
2,phil/10.2307/40231533,9a99d15f-bbe2-349e-b5af-cf96fd35c6eb,"""Tractatus"" 2.022-2.023",Raymond D. Bradley,1987,Canadian Journal of Philosophy,17,2,jstor.org/stable/10.2307/40231533,Cambridge University Press,Philosophy,
3,phil/10.2307/40230622,9af0da08-8130-3178-86cb-5df65d6fb0cd,"""Tractatus"" 5.54-5.5422",Eric B. Dayton,1976,Canadian Journal of Philosophy,6,2,jstor.org/stable/10.2307/40230622,Cambridge University Press,Philosophy,
4,phil/10.2307/40231225,041ecb2f-f8ab-398e-a1df-8813eaffa841,"'Can,' Compatibilism, and Possible Worlds",Michael J. Zimmerman,1981,Canadian Journal of Philosophy,11,4,jstor.org/stable/10.2307/40231225,Cambridge University Press,Philosophy,
...,...,...,...,...,...,...,...,...,...,...,...,...
32272,other/10.2307/44827520,12678d4f-203c-3d64-8df9-d063252866e0,ЛЕТНЯЯ РУССКАЯ ШКОЛА В ПУТНИ,Б. Ганусовский,1962,В помощь преподавателю русского языка в Америке / A Guide to Teachers of the Russian Language in America,16,63/64,jstor.org/stable/10.2307/44827520,American Councils for International Education ACTR / ACCELS,Other,Language & Literature / Slavic Studies
32273,other/10.2307/44823154,7b987d97-5881-3799-b91f-a9f096dd6240,О ЯЗЫКЕ ЗАКОНОДАТЕЛЬНЫХ АКТОВ ГЕТРА 1. /Филологические заслуги Петра 1/,М.А. ПОЛТОРАЦКАЯ,1956,В помощь преподавателю русского языка в Америке / A Guide to Teachers of the Russian Language in America,10,38,jstor.org/stable/10.2307/44823154,American Councils for International Education ACTR / ACCELS,Other,Language & Literature / Slavic Studies
32274,other/10.2307/44823659,8caa0a16-e16d-394f-8db2-af7eec824b23,РУССКИЙ ЯЗЫК ПО ТЕЛЕВИДЕНИЮ,Е.Г. АЛЕКСЕЕВА,1959,В помощь преподавателю русского языка в Америке / A Guide to Teachers of the Russian Language in America,13,50,jstor.org/stable/10.2307/44823659,American Councils for International Education ACTR / ACCELS,Other,Language & Literature / Slavic Studies
32275,other/10.2307/44823625,495a64ae-6312-3a9f-a3f1-b0ee862b0591,УПУЩЕННОЕ ВРЕМЯ,Н.П. Автономов,1958,В помощь преподавателю русского языка в Америке / A Guide to Teachers of the Russian Language in America,12,47,jstor.org/stable/10.2307/44823625,American Councils for International Education ACTR / ACCELS,Other,Language & Literature / Slavic Studies


In [13]:
df2.set_index('id').to_csv('../data/metadata.csv')

In [14]:
df2.discipline.value_counts()

discipline
Philosophy    32783
Other         32277
Literature    25343
Name: count, dtype: int64